# E6 - Spike axial: exploracion del dataset SPIDER

Este notebook explora la factibilidad tecnica de incorporar informacion axial al pipeline. El objetivo es caracterizar volumenes, mascaras, dimensiones, spacing y ejes candidatos, generando evidencia visual y tabular.

Alcance:

- No entrena modelos axiales.
- No modifica modelos sagitales.
- No usa nnU-Net.
- No integra backend.
- No realiza diagnostico clinico ni mediciones de canal/neuroforamen.
- Documenta una recomendacion preliminar de eje axial para continuar el trabajo.

In [1]:
# Dependencias para Google Colab.
try:
    import SimpleITK  # noqa: F401
except Exception:
    %pip install -q SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 10.8 MB/s eta 0:00:00


In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)

## Montaje de Drive y rutas

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

Mounted at /content/drive


In [4]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
AXIAL_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_spike_axial")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")
MULTICLASS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado")

CANDIDATES_CSV = PREPROCESS_ROOT / "E4_baseline_candidates_no_space.csv"
CONSISTENCY_CSV = PREPROCESS_ROOT / "E4_spider_consistency_summary.csv"
MAPPING_JSON = MULTICLASS_ROOT / "E5_multiclass_label_mapping.json"

for path in [AXIAL_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

if not CANDIDATES_CSV.exists():
    raise FileNotFoundError(f"No existe CSV de candidatos: {CANDIDATES_CSV}")

print("DATASET_ROOT:", DATASET_ROOT)
print("AXIAL_ROOT:", AXIAL_ROOT)
print("CONSISTENCY_CSV existe:", CONSISTENCY_CSV.exists())
print("MAPPING_JSON existe:", MAPPING_JSON.exists())

DATASET_ROOT: /content/drive/MyDrive/PFI_MVP/data/SPIDER
AXIAL_ROOT: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial
CONSISTENCY_CSV existe: True
MAPPING_JSON existe: True


## Carga y resumen de candidatos

In [5]:
def infer_case_modality(row):
    text = " ".join(str(v).lower() for v in row.values)
    if "t2_space" in text or "space" in text:
        return "t2_SPACE"
    if "t2" in text:
        return "t2"
    if "t1" in text:
        return "t1"
    return "unknown"


def ensure_case_id(df):
    out = df.copy()
    if "case_id" not in out.columns:
        for column in ["file_name", "image_path", "source_image_path", "mask_path", "source_mask_path"]:
            if column in out.columns:
                out["case_id"] = out[column].apply(lambda value: Path(str(value)).stem)
                break
    if "case_id" not in out.columns:
        raise ValueError("No se pudo construir case_id.")
    out["case_id"] = out["case_id"].astype(str)
    return out


def candidate_case_id_from_row(row):
    if "case_id" in row and pd.notna(row["case_id"]):
        return str(row["case_id"])
    for column in ["file_name", "image_path", "source_image_path", "mask_path", "source_mask_path"]:
        if column in row and pd.notna(row[column]):
            return Path(str(row[column])).stem
    return None


def first_existing_path(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_existing_dataset_path(row, kind):
    if kind not in {"image", "mask"}:
        raise ValueError("kind debe ser 'image' o 'mask'")

    case_id = candidate_case_id_from_row(row)
    columns = (
        ["image_path", "source_image_path", "image", "img_path"]
        if kind == "image"
        else ["mask_path", "source_mask_path", "mask", "seg_path"]
    )

    direct_candidates = []
    for column in columns:
        if column in row and pd.notna(row[column]):
            direct_candidates.append(Path(str(row[column])))

    base_candidates = []
    if case_id:
        if kind == "image":
            base_candidates.extend([
                DATASET_ROOT / "images" / "images" / f"{case_id}.mha",
                DATASET_ROOT / "images" / f"{case_id}.mha",
                DATASET_ROOT / f"{case_id}.mha",
            ])
        else:
            base_candidates.extend([
                DATASET_ROOT / "masks" / "masks" / f"{case_id}.mha",
                DATASET_ROOT / "masks" / f"{case_id}.mha",
                DATASET_ROOT / f"{case_id}.mha",
            ])

    found = first_existing_path(direct_candidates + base_candidates)
    if found is not None:
        return found

    if case_id and DATASET_ROOT.exists():
        matches = list(DATASET_ROOT.rglob(f"{case_id}.mha"))
        if matches:
            preferred = [path for path in matches if kind in str(path).lower()]
            return preferred[0] if preferred else matches[0]
    return None


def add_resolved_paths(df):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Resolviendo archivos"):
        item = row.to_dict()
        image_path = resolve_existing_dataset_path(row, "image")
        mask_path = resolve_existing_dataset_path(row, "mask")
        item["source_image_path"] = str(image_path) if image_path else None
        item["source_mask_path"] = str(mask_path) if mask_path else None
        item["has_existing_files"] = bool(image_path and mask_path)
        rows.append(item)
    return pd.DataFrame(rows)

In [6]:
candidates_df = ensure_case_id(pd.read_csv(CANDIDATES_CSV))
if "modality" in candidates_df.columns:
    candidates_df["inferred_modality"] = candidates_df["modality"].astype(str).str.lower()
    candidates_df.loc[candidates_df["inferred_modality"].str.contains("space", na=False), "inferred_modality"] = "t2_SPACE"
else:
    candidates_df["inferred_modality"] = candidates_df.apply(infer_case_modality, axis=1)

consistency_df = pd.read_csv(CONSISTENCY_CSV) if CONSISTENCY_CSV.exists() else None
resolved_candidates_df = add_resolved_paths(candidates_df)

summary_rows = []
for modality in ["t1", "t2", "t2_SPACE", "unknown"]:
    subset = resolved_candidates_df[resolved_candidates_df["inferred_modality"].eq(modality)]
    summary_rows.append({
        "modality": modality,
        "n_candidates": int(len(subset)),
        "n_with_existing_files": int(subset["has_existing_files"].sum()) if len(subset) else 0,
        "included_in_initial_spike": bool(modality in {"t1", "t2"}),
        "note": "t2_SPACE excluido inicialmente si aparece" if modality == "t2_SPACE" else "",
    })

candidate_summary_df = pd.DataFrame(summary_rows)
candidate_summary_csv_path = AXIAL_ROOT / "E6_axial_candidate_summary.csv"
candidate_summary_df.to_csv(candidate_summary_csv_path, index=False)

print("Total candidatos:", len(candidates_df))
print("Resumen candidatos:", candidate_summary_csv_path)
display(candidate_summary_df)
if consistency_df is not None:
    print("Consistency summary cargado:", CONSISTENCY_CSV, "filas:", len(consistency_df))

Resolviendo archivos:   0%|          | 0/406 [00:00<?, ?it/s]

Total candidatos: 406
Resumen candidatos: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_candidate_summary.csv


,modality,n_candidates,n_with_existing_files,included_in_initial_spike,note
0,t1,196,196,True,
1,t2,210,210,True,
2,t2_SPACE,0,0,False,t2_SPACE excluido inicialmente si aparece
3,unknown,0,0,False,


Consistency summary cargado: /content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento/E4_spider_consistency_summary.csv filas: 447


## Seleccion de muestra exploratoria

In [7]:
RANDOM_SEED = 42
N_T1_SAMPLE = 10
N_T2_SAMPLE = 10

eligible_df = resolved_candidates_df[
    resolved_candidates_df["has_existing_files"]
    & resolved_candidates_df["inferred_modality"].isin(["t1", "t2"])
].copy()

selected_parts = []
for modality, n_sample in [("t1", N_T1_SAMPLE), ("t2", N_T2_SAMPLE)]:
    subset = eligible_df[eligible_df["inferred_modality"].eq(modality)]
    if len(subset) > 0:
        selected_parts.append(subset.sample(n=min(n_sample, len(subset)), random_state=RANDOM_SEED))

selected_cases_df = (
    pd.concat(selected_parts, ignore_index=True)
    if selected_parts else pd.DataFrame(columns=eligible_df.columns)
).sort_values(["inferred_modality", "case_id"]).reset_index(drop=True)

selected_cases_csv_path = AXIAL_ROOT / "E6_axial_selected_cases.csv"
selected_cases_df.to_csv(selected_cases_csv_path, index=False)

print("Casos seleccionados:", len(selected_cases_df))
print("CSV seleccion:", selected_cases_csv_path)
display(selected_cases_df[["case_id", "inferred_modality", "source_image_path", "source_mask_path"]].head(25))

if len(selected_cases_df) == 0:
    raise RuntimeError("No hay casos T1/T2 con archivos existentes para el spike axial.")

Casos seleccionados: 20
CSV seleccion: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_selected_cases.csv


,case_id,inferred_modality,source_image_path,source_mask_path
0,11_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
1,122_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
2,171_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
3,187_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
4,192_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
5,237_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
6,34_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
7,52_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
8,67_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
9,86_t1,t1,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...


## Lectura MHA, metadata y mapping opcional

In [8]:
label_group_mapping = None
if MAPPING_JSON.exists():
    with open(MAPPING_JSON, "r", encoding="utf-8") as f:
        label_group_mapping = {int(k): int(v) for k, v in json.load(f).items()}


def read_mha_with_metadata(path):
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)  # Formato numpy: z, y, x
    metadata = {
        "spacing_xyz": tuple(float(v) for v in image.GetSpacing()),
        "origin_xyz": tuple(float(v) for v in image.GetOrigin()),
        "direction": tuple(float(v) for v in image.GetDirection()),
    }
    return image, array, metadata


def metadata_close(a, b, atol=1e-4):
    return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=atol))


def validate_image_mask_pair(image_meta, mask_meta, image_shape, mask_shape):
    return {
        "shape_match": tuple(image_shape) == tuple(mask_shape),
        "spacing_match": metadata_close(image_meta["spacing_xyz"], mask_meta["spacing_xyz"]),
        "origin_match": metadata_close(image_meta["origin_xyz"], mask_meta["origin_xyz"]),
        "direction_match": metadata_close(image_meta["direction"], mask_meta["direction"]),
    }


def map_mask_grouped(mask):
    if label_group_mapping is None:
        return None
    grouped = np.zeros_like(mask, dtype=np.int16)
    for label in np.unique(mask):
        label_int = int(label)
        grouped[mask == label_int] = int(label_group_mapping.get(label_int, 0))
    return grouped


def take_slice(array, axis, index):
    return np.take(array, indices=int(index), axis=int(axis))


def normalize_for_display(slice_2d, p_low=1, p_high=99):
    values = slice_2d.astype(np.float32)
    low, high = np.percentile(values, [p_low, p_high])
    if np.isclose(low, high):
        return np.zeros_like(values, dtype=np.float32)
    return np.clip((values - low) / (high - low), 0, 1).astype(np.float32)


def binary_mask(mask):
    return (mask > 0).astype(np.uint8)

## Analisis de ejes

In [9]:
def axis_slice_foreground_stats(mask_binary, axis):
    n_slices = mask_binary.shape[axis]
    areas = np.asarray([
        int(np.sum(take_slice(mask_binary, axis, idx) > 0))
        for idx in range(n_slices)
    ], dtype=np.int64)
    max_index = int(np.argmax(areas)) if len(areas) else 0
    return {
        "n_slices": int(n_slices),
        "max_foreground_slice_index": max_index,
        "max_foreground_area": int(areas[max_index]) if len(areas) else 0,
        "n_slices_with_foreground": int(np.sum(areas > 0)),
        "foreground_area_mean": float(np.mean(areas)) if len(areas) else 0.0,
    }


case_cache = {}
axis_rows = []
orientation_issue_rows = []

for _, row in tqdm(selected_cases_df.iterrows(), total=len(selected_cases_df), desc="Analizando ejes"):
    case_id = str(row["case_id"])
    image_path = Path(row["source_image_path"])
    mask_path = Path(row["source_mask_path"])
    image_itk, image_arr, image_meta = read_mha_with_metadata(image_path)
    mask_itk, mask_arr, mask_meta = read_mha_with_metadata(mask_path)
    pair_checks = validate_image_mask_pair(image_meta, mask_meta, image_arr.shape, mask_arr.shape)
    mask_bin = binary_mask(mask_arr)
    mask_grouped = map_mask_grouped(mask_arr)

    case_cache[case_id] = {
        "row": row.to_dict(),
        "image": image_arr,
        "mask": mask_arr,
        "mask_binary": mask_bin,
        "mask_grouped": mask_grouped,
        "image_meta": image_meta,
        "mask_meta": mask_meta,
        "pair_checks": pair_checks,
    }

    if not all(pair_checks.values()):
        orientation_issue_rows.append({
            "case_id": case_id,
            "inferred_modality": row["inferred_modality"],
            **pair_checks,
        })

    for axis in [0, 1, 2]:
        stats = axis_slice_foreground_stats(mask_bin, axis)
        axis_rows.append({
            "case_id": case_id,
            "inferred_modality": row["inferred_modality"],
            "shape_zyx": str(tuple(int(v) for v in image_arr.shape)),
            "spacing_xyz": str(tuple(float(v) for v in image_meta["spacing_xyz"])),
            "axis": int(axis),
            **stats,
            **pair_checks,
            "source_image_path": str(image_path),
            "source_mask_path": str(mask_path),
        })

axis_analysis_df = pd.DataFrame(axis_rows)
axis_analysis_csv_path = AXIAL_ROOT / "E6_axial_axis_analysis.csv"
axis_analysis_df.to_csv(axis_analysis_csv_path, index=False)

orientation_issues_df = pd.DataFrame(orientation_issue_rows)
orientation_issues_csv_path = AXIAL_ROOT / "E6_axial_orientation_issues.csv"
if len(orientation_issues_df) > 0:
    orientation_issues_df.to_csv(orientation_issues_csv_path, index=False)

print("Analisis por eje:", axis_analysis_csv_path)
print("Casos con problemas de orientacion/metadata:", len(orientation_issues_df))
display(axis_analysis_df.head(12))

Analizando ejes:   0%|          | 0/20 [00:00<?, ?it/s]

Analisis por eje: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_axis_analysis.csv
Casos con problemas de orientacion/metadata: 0


,case_id,inferred_modality,shape_zyx,spacing_xyz,axis,n_slices,max_foreground_slice_index,max_foreground_area,n_slices_with_foreground,foreground_area_mean,shape_match,spacing_match,origin_match,direction_match,source_image_path,source_mask_path
0,11_t1,t1,"(590, 512, 21)","(3.312444001646348, 0.5859375000028706, 0.5109...",0,590,183,1338,496,744.311864,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
1,11_t1,t1,"(590, 512, 21)","(3.312444001646348, 0.5859375000028706, 0.5109...",1,512,236,3619,227,857.703125,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
2,11_t1,t1,"(590, 512, 21)","(3.312444001646348, 0.5859375000028706, 0.5109...",2,21,10,45045,20,20911.619048,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
3,122_t1,t1,"(320, 320, 15)","(4.8000000544956425, 0.875, 0.875)",0,320,135,452,244,212.890625,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
4,122_t1,t1,"(320, 320, 15)","(4.8000000544956425, 0.875, 0.875)",1,320,171,1379,104,212.890625,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
5,122_t1,t1,"(320, 320, 15)","(4.8000000544956425, 0.875, 0.875)",2,15,5,10988,15,4541.666667,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
6,171_t1,t1,"(384, 384, 15)","(4.399999855318704, 0.72916668653488, 0.729166...",0,384,100,846,338,493.809896,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
7,171_t1,t1,"(384, 384, 15)","(4.399999855318704, 0.72916668653488, 0.729166...",1,384,228,2671,136,493.809896,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
8,171_t1,t1,"(384, 384, 15)","(4.399999855318704, 0.72916668653488, 0.729166...",2,15,6,26635,15,12641.533333,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
9,187_t1,t1,"(748, 465, 21)","(4.381844504510948, 0.6681325938412357, 0.4129...",0,748,220,679,620,361.155080,True,True,True,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...


## Visualizacion multiplanar

In [10]:
def overlay_mask(ax, mask_2d, color=(1.0, 0.1, 0.1), alpha=0.45):
    rgba = np.zeros((*mask_2d.shape, 4), dtype=np.float32)
    rgba[mask_2d > 0, :3] = color
    rgba[mask_2d > 0, 3] = alpha
    ax.imshow(rgba, interpolation="nearest")


def representative_index_for_axis(case_id, axis):
    subset = axis_analysis_df[(axis_analysis_df["case_id"].eq(case_id)) & (axis_analysis_df["axis"].eq(axis))]
    if len(subset) == 0:
        return int(case_cache[case_id]["image"].shape[axis] // 2)
    row = subset.iloc[0]
    if int(row["max_foreground_area"]) > 0:
        return int(row["max_foreground_slice_index"])
    return int(case_cache[case_id]["image"].shape[axis] // 2)


def plot_multiplanar_case(case_id, output_path):
    cached = case_cache[case_id]
    image = cached["image"]
    mask = cached["mask_binary"]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for col, axis in enumerate([0, 1, 2]):
        idx = representative_index_for_axis(case_id, axis)
        image_slice = normalize_for_display(take_slice(image, axis, idx))
        mask_slice = take_slice(mask, axis, idx)
        axes[col].imshow(image_slice, cmap="gray", interpolation="nearest")
        overlay_mask(axes[col], mask_slice)
        axes[col].set_title(f"Eje {axis} - slice {idx}")
        axes[col].axis("off")

    modality = cached["row"].get("inferred_modality", "unknown")
    fig.suptitle(f"{case_id} ({modality}) - multiplanar z,y,x")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


multiplanar_figure_paths = []
for idx, case_id in enumerate(list(case_cache.keys())[:3], start=1):
    output_path = FIGURES_ROOT / f"E6_multiplanar_example_{idx:02d}.png"
    plot_multiplanar_case(case_id, output_path)
    multiplanar_figure_paths.append(str(output_path))

print("Figuras multiplanares:")
for path in multiplanar_figure_paths:
    print(path)

Figuras multiplanares:
/content/drive/MyDrive/PFI_MVP/figures/E6_multiplanar_example_01.png
/content/drive/MyDrive/PFI_MVP/figures/E6_multiplanar_example_02.png
/content/drive/MyDrive/PFI_MVP/figures/E6_multiplanar_example_03.png


## Heuristica preliminar de eje axial

In [11]:
def recommend_axial_axis(shape_zyx, spacing_xyz, axis_stats):
    # SimpleITK entrega arrays z,y,x. En muchos volumenes, recorrer z produce cortes x-y,
    # que suelen corresponder a una vista axial. No se asume sin revisar spacing y evidencia.
    spacing_zyx = (float(spacing_xyz[2]), float(spacing_xyz[1]), float(spacing_xyz[0]))
    shape_zyx = tuple(int(v) for v in shape_zyx)

    reasons = []
    confidence_score = 0

    axis0_spacing = spacing_zyx[0]
    in_plane_spacing_similarity = abs(spacing_zyx[1] - spacing_zyx[2]) / max(spacing_zyx[1], spacing_zyx[2], 1e-8)
    if in_plane_spacing_similarity < 0.25:
        confidence_score += 1
        reasons.append("spacing y/x similar; cortes sobre eje 0 preservan plano y-x")
    if axis0_spacing >= max(spacing_zyx[1], spacing_zyx[2]):
        confidence_score += 1
        reasons.append("spacing del eje z no es menor que el in-plane")
    if shape_zyx[0] <= max(shape_zyx[1], shape_zyx[2]):
        confidence_score += 1
        reasons.append("eje 0 tiene cantidad de slices compatible con stack")
    if axis_stats.get(0, {}).get("n_slices_with_foreground", 0) > 0:
        confidence_score += 1
        reasons.append("hay foreground al recorrer eje 0")

    if confidence_score >= 3:
        confidence = "alta"
    elif confidence_score == 2:
        confidence = "media"
    else:
        confidence = "baja"

    return {
        "candidate_axial_axis": 0,
        "reason": "; ".join(reasons) if reasons else "heuristica preliminar z,y,x insuficiente; requiere revision visual",
        "confidence": confidence,
        "spacing_zyx": str(spacing_zyx),
    }


recommendation_rows = []
for case_id, cached in case_cache.items():
    image = cached["image"]
    spacing_xyz = cached["image_meta"]["spacing_xyz"]
    axis_stats = {}
    for axis in [0, 1, 2]:
        subset = axis_analysis_df[(axis_analysis_df["case_id"].eq(case_id)) & (axis_analysis_df["axis"].eq(axis))]
        axis_stats[axis] = subset.iloc[0].to_dict() if len(subset) else {}
    rec = recommend_axial_axis(image.shape, spacing_xyz, axis_stats)
    recommendation_rows.append({
        "case_id": case_id,
        "inferred_modality": cached["row"].get("inferred_modality", "unknown"),
        "shape_zyx": str(tuple(int(v) for v in image.shape)),
        "spacing_xyz": str(tuple(float(v) for v in spacing_xyz)),
        **rec,
    })

axis_recommendation_df = pd.DataFrame(recommendation_rows)
axis_recommendation_csv_path = AXIAL_ROOT / "E6_candidate_axial_axis_recommendation.csv"
axis_recommendation_df.to_csv(axis_recommendation_csv_path, index=False)

print("Recomendacion eje axial:", axis_recommendation_csv_path)
display(axis_recommendation_df)

Recomendacion eje axial: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_candidate_axial_axis_recommendation.csv


,case_id,inferred_modality,shape_zyx,spacing_xyz,candidate_axial_axis,reason,confidence,spacing_zyx
0,11_t1,t1,"(590, 512, 21)","(3.312444001646348, 0.5859375000028706, 0.5109...",0,hay foreground al recorrer eje 0,baja,"(0.510918381957822, 0.5859375000028706, 3.3124..."
1,122_t1,t1,"(320, 320, 15)","(4.8000000544956425, 0.875, 0.875)",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.875, 0.875, 4.8000000544956425)"
2,171_t1,t1,"(384, 384, 15)","(4.399999855318704, 0.72916668653488, 0.729166...",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.72916668653488, 0.72916668653488, 4.3999998..."
3,187_t1,t1,"(748, 465, 21)","(4.381844504510948, 0.6681325938412357, 0.4129...",0,hay foreground al recorrer eje 0,baja,"(0.4129231489837082, 0.6681325938412357, 4.381..."
4,192_t1,t1,"(448, 448, 24)","(3.2999548400246894, 0.625, 0.6252388405966087)",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.6252388405966087, 0.625, 3.2999548400246894)"
5,237_t1,t1,"(611, 646, 18)","(4.397158886379771, 0.47702580399344185, 0.504...",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.5048838230591457, 0.47702580399344185, 4.39..."
6,34_t1,t1,"(17, 512, 512)","(0.5859375, 0.5859375, 3.29999995231625)",0,spacing y/x similar; cortes sobre eje 0 preser...,alta,"(3.29999995231625, 0.5859375, 0.5859375)"
7,52_t1,t1,"(17, 512, 512)","(0.5859375, 0.5859375, 3.299999961969196)",0,spacing y/x similar; cortes sobre eje 0 preser...,alta,"(3.299999961969196, 0.5859375, 0.5859375)"
8,67_t1,t1,"(329, 896, 23)","(4.324627730240991, 0.3125000000015348, 0.8690...",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.8690515293266543, 0.3125000000015348, 4.324..."
9,86_t1,t1,"(424, 448, 25)","(3.292637052174925, 0.625, 0.662695647310926)",0,eje 0 tiene cantidad de slices compatible con ...,media,"(0.662695647310926, 0.625, 3.292637052174925)"


## Overlays axiales candidatos

In [12]:
def plot_single_overlay(image_2d, mask_2d, title, output_path):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(normalize_for_display(image_2d), cmap="gray", interpolation="nearest")
    overlay_mask(ax, mask_2d)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


axial_overlay_paths = []
for idx, row in enumerate(axis_recommendation_df.head(5).itertuples(index=False), start=1):
    case_id = row.case_id
    axis = int(row.candidate_axial_axis)
    cached = case_cache[case_id]
    slice_idx = representative_index_for_axis(case_id, axis)
    image_slice = take_slice(cached["image"], axis, slice_idx)
    mask_slice = take_slice(cached["mask_binary"], axis, slice_idx)
    output_path = FIGURES_ROOT / f"E6_axial_candidate_overlay_{idx:02d}.png"
    plot_single_overlay(
        image_slice,
        mask_slice,
        f"{case_id} - eje axial candidato {axis}, slice {slice_idx}",
        output_path,
    )
    axial_overlay_paths.append(str(output_path))

print("Overlays axiales candidatos:")
for path in axial_overlay_paths:
    print(path)

Overlays axiales candidatos:
/content/drive/MyDrive/PFI_MVP/figures/E6_axial_candidate_overlay_01.png
/content/drive/MyDrive/PFI_MVP/figures/E6_axial_candidate_overlay_02.png
/content/drive/MyDrive/PFI_MVP/figures/E6_axial_candidate_overlay_03.png
/content/drive/MyDrive/PFI_MVP/figures/E6_axial_candidate_overlay_04.png
/content/drive/MyDrive/PFI_MVP/figures/E6_axial_candidate_overlay_05.png


## Comparacion sagital vs axial candidata

In [13]:
SAGITTAL_AXIS_E5 = 2


def plot_sagittal_vs_axial(case_id, output_path):
    cached = case_cache[case_id]
    image = cached["image"]
    mask = cached["mask_binary"]
    axial_axis = int(axis_recommendation_df[axis_recommendation_df["case_id"].eq(case_id)]["candidate_axial_axis"].iloc[0])
    axial_idx = representative_index_for_axis(case_id, axial_axis)
    sagittal_idx = representative_index_for_axis(case_id, SAGITTAL_AXIS_E5)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    sag_image = take_slice(image, SAGITTAL_AXIS_E5, sagittal_idx)
    sag_mask = take_slice(mask, SAGITTAL_AXIS_E5, sagittal_idx)
    ax_image = take_slice(image, axial_axis, axial_idx)
    ax_mask = take_slice(mask, axial_axis, axial_idx)

    axes[0].imshow(normalize_for_display(sag_image), cmap="gray", interpolation="nearest")
    overlay_mask(axes[0], sag_mask)
    axes[0].set_title(f"Sagital E5 eje {SAGITTAL_AXIS_E5} - slice {sagittal_idx}")
    axes[0].axis("off")

    axes[1].imshow(normalize_for_display(ax_image), cmap="gray", interpolation="nearest")
    overlay_mask(axes[1], ax_mask, color=(0.1, 0.45, 1.0))
    axes[1].set_title(f"Axial candidato eje {axial_axis} - slice {axial_idx}")
    axes[1].axis("off")

    fig.suptitle(f"{case_id} - vistas complementarias")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


sagittal_vs_axial_paths = []
for idx, case_id in enumerate(list(case_cache.keys())[:3], start=1):
    output_path = FIGURES_ROOT / f"E6_sagital_vs_axial_example_{idx:02d}.png"
    plot_sagittal_vs_axial(case_id, output_path)
    sagittal_vs_axial_paths.append(str(output_path))

print("Comparaciones sagital vs axial:")
for path in sagittal_vs_axial_paths:
    print(path)

Comparaciones sagital vs axial:
/content/drive/MyDrive/PFI_MVP/figures/E6_sagital_vs_axial_example_01.png
/content/drive/MyDrive/PFI_MVP/figures/E6_sagital_vs_axial_example_02.png
/content/drive/MyDrive/PFI_MVP/figures/E6_sagital_vs_axial_example_03.png


## Reporte JSON y conclusion Markdown

In [14]:
modalities_included = sorted(selected_cases_df["inferred_modality"].dropna().unique().tolist())
modality_counts = selected_cases_df["inferred_modality"].value_counts().to_dict()
axis_mode = int(axis_recommendation_df["candidate_axial_axis"].mode().iloc[0])
confidence_counts = axis_recommendation_df["confidence"].value_counts().to_dict()
confidence_percentages = {
    key: float(value / max(len(axis_recommendation_df), 1))
    for key, value in confidence_counts.items()
}

exported_files = {
    "candidate_summary": str(candidate_summary_csv_path),
    "selected_cases": str(selected_cases_csv_path),
    "axis_analysis": str(axis_analysis_csv_path),
    "orientation_issues": str(orientation_issues_csv_path) if len(orientation_issues_df) > 0 else None,
    "axis_recommendation": str(axis_recommendation_csv_path),
    "multiplanar_figures": multiplanar_figure_paths,
    "axial_candidate_overlays": axial_overlay_paths,
    "sagittal_vs_axial_figures": sagittal_vs_axial_paths,
}

preliminary_conclusion = (
    "El spike sugiere que es factible extraer vistas axiales candidatas desde los volumenes SPIDER "
    "usando el eje 0 del array z,y,x como hipotesis inicial, sujeto a revision visual y validacion con especialistas. "
    "El plano axial debe tratarse como complementario al sagital para proximas etapas."
)

report = {
    "n_cases_analyzed": int(len(selected_cases_df)),
    "modalities_included": modalities_included,
    "modality_counts": {str(k): int(v) for k, v in modality_counts.items()},
    "excluded_t2_space_count": int(len(resolved_candidates_df[resolved_candidates_df["inferred_modality"].eq("t2_SPACE")])),
    "axes_evaluated": [0, 1, 2],
    "most_frequent_recommended_axial_axis": axis_mode,
    "recommendation_confidence_counts": {str(k): int(v) for k, v in confidence_counts.items()},
    "recommendation_confidence_percentages": confidence_percentages,
    "n_cases_with_orientation_issues": int(len(orientation_issues_df)),
    "exported_files": exported_files,
    "preliminary_conclusion": preliminary_conclusion,
}

report_json_path = AXIAL_ROOT / "E6_axial_spike_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte JSON:", report_json_path)
print(json.dumps(report, indent=2, ensure_ascii=False)[:4000])

Reporte JSON: /content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_spike_report.json
{
  "n_cases_analyzed": 20,
  "modalities_included": [
    "t1",
    "t2"
  ],
  "modality_counts": {
    "t1": 10,
    "t2": 10
  },
  "excluded_t2_space_count": 0,
  "axes_evaluated": [
    0,
    1,
    2
  ],
  "most_frequent_recommended_axial_axis": 0,
  "recommendation_confidence_counts": {
    "media": 11,
    "alta": 5,
    "baja": 4
  },
  "recommendation_confidence_percentages": {
    "media": 0.55,
    "alta": 0.25,
    "baja": 0.2
  },
  "n_cases_with_orientation_issues": 0,
  "exported_files": {
    "candidate_summary": "/content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_candidate_summary.csv",
    "selected_cases": "/content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_selected_cases.csv",
    "axis_analysis": "/content/drive/MyDrive/PFI_MVP/results/E6_spike_axial/E6_axial_axis_analysis.csv",
    "orientation_issues": null,
    "axis_recommendation": "/conte

In [15]:
shape_summary_md = (
    axis_analysis_df[["case_id", "inferred_modality", "shape_zyx", "spacing_xyz"]]
    .drop_duplicates()
    .head(20)
    .to_markdown(index=False)
)
axis_recommendation_md = axis_recommendation_df.to_markdown(index=False)
candidate_summary_md = candidate_summary_df.to_markdown(index=False)

conclusion_text = f'''# Conclusion tecnica - E6 Spike axial exploratorio

## Objetivo

Explorar la factibilidad tecnica de incorporar informacion axial al pipeline sobre SPIDER, analizando volumenes, mascaras, shapes, spacing y ejes disponibles en arrays `z,y,x`.

## Relacion con user research

Los profesionales entrevistados remarcaron que el plano axial es necesario para evaluar canal, neuroforamenes y compromiso radicular. Este spike no implementa esas mediciones ni emite diagnosticos; solo documenta si el dataset permite construir evidencia axial preliminar.

## Configuracion

- Dataset: SPIDER.
- Candidatos base: `{CANDIDATES_CSV}`.
- Modalidades incluidas inicialmente: {modalities_included}.
- Casos analizados: {len(selected_cases_df)}.
- t2_SPACE excluidos inicialmente: {report["excluded_t2_space_count"]}.
- Ejes evaluados: 0, 1 y 2 sobre arrays `z,y,x`.

## Resumen de candidatos

{candidate_summary_md}

## Hallazgos sobre shapes, spacing y ejes

{shape_summary_md}

## Eje axial recomendado preliminar

El eje axial candidato mas frecuente fue el eje `{axis_mode}`. La heuristica considera que SimpleITK devuelve arrays en formato `z,y,x`, por lo que recorrer el eje 0 produce cortes `y,x`, usualmente compatibles con una vista axial. Esta inferencia se mantiene como preliminar y requiere revision visual.

{axis_recommendation_md}

## Ejemplos visuales generados

- Multiplanar: {multiplanar_figure_paths}
- Overlays axiales candidatos: {axial_overlay_paths}
- Sagital vs axial: {sagittal_vs_axial_paths}

## Limitaciones

- Esta etapa no entrena modelos.
- La identificacion de eje axial es preliminar.
- Las mascaras de SPIDER pueden no estar disenadas especificamente para evaluar hallazgos clinicos axiales.
- Puede requerirse dataset axial especifico o validacion con especialistas.
- No se hacen mediciones de canal ni neuroforamen todavia.
- No se incorpora diagnostico.
- t2_SPACE queda documentado, pero excluido inicialmente si introduce problemas de orientacion.

## Recomendacion para siguiente paso

Continuar con una revision visual asistida de los ejemplos exportados, confirmar el eje axial con especialistas o documentacion del dataset, y luego definir si conviene implementar un baseline axial 2D exploratorio o buscar un dataset axial mas adecuado para canal/neuroforamenes.

Este notebook no reemplaza criterio profesional y no emite diagnosticos ni recomendaciones terapeuticas.
'''

conclusion_path = DOCS_ROOT / "E6_spike_axial_exploracion_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion Markdown:", conclusion_path)

Conclusion Markdown: /content/drive/MyDrive/PFI_MVP/docs/E6_spike_axial_exploracion_conclusion.md


## Criterio de aceptacion

El notebook cumple el objetivo si:

- Carga candidatos de E4.
- Analiza shapes, spacing y ejes.
- Exporta analisis por eje.
- Genera visualizaciones multiplanares.
- Propone un eje axial candidato con justificacion.
- Exporta overlays axiales.
- Deja una conclusion clara sobre la factibilidad preliminar del plano axial.

No se entrenan modelos axiales en este notebook.